In [1]:
import pandas as pd

X_train = pd.read_parquet('../data/X_train.parquet')
X_test = pd.read_parquet('../data/X_test.parquet')
y_train = pd.read_parquet('../data/y_train.parquet')['churn']
y_test = pd.read_parquet('../data/y_test.parquet')['churn']

X_train.shape, X_test.shape

((8000, 12), (2000, 12))

In [2]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

d:\CustomerIQ\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:455: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
d:\CustomerIQ\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression(max_iter=1000)

In [3]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [4]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)

d:\CustomerIQ\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:455: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


LogisticRegression(max_iter=1000)

In [5]:
from sklearn.metrics import classification_report

y_pred = model.predict(X_test_scaled)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.82      0.97      0.89      1593
           1       0.58      0.19      0.28       407

    accuracy                           0.81      2000
   macro avg       0.70      0.58      0.59      2000
weighted avg       0.77      0.81      0.77      2000



In [6]:
model_balanced = LogisticRegression(max_iter=1000, class_weight='balanced')
model_balanced.fit(X_train_scaled, y_train)

y_pred_balanced = model_balanced.predict(X_test_scaled)
print(classification_report(y_test, y_pred_balanced))

              precision    recall  f1-score   support

           0       0.90      0.72      0.80      1593
           1       0.39      0.71      0.50       407

    accuracy                           0.71      2000
   macro avg       0.65      0.71      0.65      2000
weighted avg       0.80      0.71      0.74      2000



d:\CustomerIQ\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:455: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

rf_model = RandomForestClassifier(random_state=42, class_weight='balanced')
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)
print(classification_report(y_test, y_pred_rf))

              precision    recall  f1-score   support

           0       0.87      0.97      0.92      1593
           1       0.80      0.45      0.57       407

    accuracy                           0.86      2000
   macro avg       0.84      0.71      0.75      2000
weighted avg       0.86      0.86      0.85      2000



In [8]:
import pandas as pd

importances = pd.Series(rf_model.feature_importances_, index=X_train.columns).sort_values(ascending=False)
importances

age                 0.250673
estimated_salary    0.137645
products_number     0.134518
credit_score        0.131181
balance             0.129106
tenure              0.080836
active_member       0.036143
country_Germany     0.031089
gender_Male         0.021308
credit_card         0.018019
country_Spain       0.015108
has_zero_balance    0.014374
dtype: float64

In [9]:
from sklearn.inspection import permutation_importance

perm_result = permutation_importance(rf_model, X_test, y_test, n_repeats=10, random_state=42)

perm_importances = pd.Series(perm_result.importances_mean, index=X_test.columns).sort_values(ascending=False)
perm_importances

age                 0.06600
products_number     0.05865
active_member       0.02475
country_Germany     0.01500
balance             0.01115
tenure              0.00435
estimated_salary    0.00275
country_Spain       0.00150
gender_Male         0.00135
has_zero_balance    0.00115
credit_score        0.00105
credit_card        -0.00115
dtype: float64

In [10]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(random_state=42, scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum())
xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict(X_test)
print(classification_report(y_test, y_pred_xgb))

              precision    recall  f1-score   support

           0       0.90      0.88      0.89      1593
           1       0.56      0.61      0.59       407

    accuracy                           0.82      2000
   macro avg       0.73      0.75      0.74      2000
weighted avg       0.83      0.82      0.83      2000



In [11]:
xgb_importances = pd.Series(xgb_model.feature_importances_, index=X_train.columns).sort_values(ascending=False)
xgb_importances

products_number     0.221660
has_zero_balance    0.197684
active_member       0.124897
age                 0.096463
country_Germany     0.084747
gender_Male         0.050043
country_Spain       0.047830
balance             0.047537
credit_score        0.035894
estimated_salary    0.032336
credit_card         0.030614
tenure              0.030297
dtype: float32